# Clean HubDailyEvents

Cleans a raw `HubDailyEventData_yyyy-mm-dd.csv` export.

**Deviations / interpretations, mirroring the approach in the other three notebooks:**
- *"Remove completely blank rows"* is evaluated against the `LS_COLS` fields present in the raw data, not the full raw column set — same reasoning as the other notebooks. In this dataset the 2 completely-blank rows are exactly the 2 rows with a blank `Date`, so the next rule (dropping blank `Date`) removes them too.
- Only rows with a blank `Date` are dropped, per the notes (no `CompanyCode` condition here) — matching the same rule just applied to `Clean_HubDailyUsers.ipynb`. About 1,842 rows have a blank `CompanyCode` with `Date` still populated; those are kept.
- `Feature` has no source column anywhere in the raw data. Per the notes, it's inserted as a new, always-blank column positioned between `Operation` and `EventAction` (matching its position in `LS_COLS`), rather than being derived from anything.
- The notes name this column `SessionswithEvent` in the `LS_COLS` list but `SessionsWithEvent` in the `LS_INT_COLS` list — the raw column is actually `SessionsWithEvent` (capital `W`), which is the casing used throughout this notebook.
- The rename step (`eventAction` → `EventAction`, etc.) is applied first, so the later "strip HTML" and "trim whitespace" steps — which the notes describe using the raw lowercase column names — are applied to their renamed equivalents (the same underlying columns).
- HTML stripping on `EventSection` runs *before* the whitespace-trim/collapse pass, so any double spaces left behind by removing tags (e.g. `"<p>A</p>  <p>B</p>"` → `"A  B"`) get collapsed to a single space along with genuine source whitespace, rather than being missed.
- The whitespace step does two things per the notes: `.str.strip()` for leading/trailing whitespace, and a separate collapse of any internal run of whitespace (double spaces, tabs, ...) down to a single space. The raw data has real cases of both (e.g. `CompanyName` has 85 rows with a double space and 79 with a tab; `eventLabel` has 5 rows with a double space).
- The raw text is pre-processed before parsing to protect a literal backslash in the data (the real company name `TBWA\RAAD`) from being stripped by `escapechar` — see the note on the read-CSV cell below.
- The file is read and written with `encoding="utf-8"` explicitly, matching the other notebooks' reasoning (avoiding `cp1252` corruption of accented characters).


## Imports

In [1]:
import csv
import html
import io
import re
from pathlib import Path

import pandas as pd


## Schema constants

- `LS_COLS` — the final column set and order for the cleaned output.
- `LS_STRING_COLS` — the free-text columns that get whitespace-trimmed and have internal double-whitespace collapsed to a single space.
- `LS_INT_COLS` — the numeric columns that get cast to integer type.
- `HTML_COLS` — the columns that get HTML tags/entities stripped.
- `RENAME_MAP` — raw → cleaned column renames (see the note at the top of this notebook).
- `SORT_COLS` — the columns (and order) used for the final ascending sort (the notes call this `sort_order`).
- `FILENAME_RE` — extracts the year/month from the input filename, used both to build `month_tag` and to auto-detect the input file.
- `GROUP_COLS`/`SUM_COLS` — used to collapse duplicate rows (see "Collapse duplicate rows" below): every `LS_COLS` field except `SUM_COLS` is a group-by key. `SUM_COLS` is `LS_INT_COLS` plus `Users`, which — despite being a measure — isn't in `LS_INT_COLS`/`LS_STRING_COLS` and so is never cast to int by `clean()`; `collapse_duplicates` casts it defensively before summing.


In [2]:
LS_COLS = [
    "Date", "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode",
    "Operation", "Feature", "EventAction", "EventCategory", "EventLabel", "EventSection",
    "EventQuestion", "UserType", "Users", "TotalEvents", "UniqueEvents",
    "SessionsWithEvent", "Events/SessionwithEvent",
]
LS_STRING_COLS = [
    "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode", "Operation",
    "EventAction", "EventCategory", "EventLabel", "EventSection", "EventQuestion", "UserType",
]
LS_INT_COLS = ["TotalEvents", "UniqueEvents", "SessionsWithEvent", "Events/SessionwithEvent"]
HTML_COLS = ["EventSection"]

RENAME_MAP = {
    "eventAction": "EventAction",
    "eventCategory": "EventCategory",
    "eventLabel": "EventLabel",
    "eventSection": "EventSection",
    "eventQuestion": "EventQuestion",
    "EventsPerSessionWithEvent": "Events/SessionwithEvent",
}

SORT_COLS = ["Date", "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode"]

FILENAME_RE = re.compile(r"^HubDailyEventData_(\d{4})-(\d{2})-\d{2}\.csv$")

# Duplicate rows are collapsed by grouping on every LS_COLS field except the summed
# measures and summing those. Users isn't in LS_INT_COLS (clean() never casts it), so
# collapse_duplicates casts it defensively before summing.
SUM_COLS = ["Users"] + LS_INT_COLS
GROUP_COLS = [c for c in LS_COLS if c not in SUM_COLS]


## Locate the input file

`find_default_input` looks for a single `HubDailyEventData_yyyy-mm-dd.csv` file in a given directory and returns it automatically. If none or several are found, it raises rather than silently guessing which one to use.


In [3]:
def find_default_input(directory: Path) -> Path:
    matches = sorted(p for p in directory.glob("HubDailyEventData_*.csv") if FILENAME_RE.match(p.name))
    if not matches:
        raise FileNotFoundError(f"No HubDailyEventData_yyyy-mm-dd.csv file found in {directory}")
    if len(matches) > 1:
        raise ValueError(
            f"Multiple candidate input files found in {directory}: "
            f"{[m.name for m in matches]}. Pass one explicitly."
        )
    return matches[0]


## Derive `month_tag` from the filename

The output name has the format of `HubDailyEvents_<month_tag>_cleaned.csv`, where `month_tag` is `yyyymm` for the month *before* the input filename's `yyyy-mm-dd` date suffix (the export date's month minus one), per the notes' worked example.


In [4]:
def month_tag_from_filename(path: Path) -> str:
    match = FILENAME_RE.match(path.name)
    if not match:
        raise ValueError(f"Filename '{path.name}' does not match expected pattern HubDailyEventData_yyyy-mm-dd.csv")
    year, month = (int(g) for g in match.groups())
    # month_tag refers to the prior month's data, not the export date's month.
    year, month = (year - 1, 12) if month == 1 else (year, month - 1)
    return f"{year}{month:02d}"


## Cleaning logic

The core transformation, in the order implemented (the notes list these unordered):

1. Rename `eventAction`→`EventAction`, `eventCategory`→`EventCategory`, `eventLabel`→`EventLabel`, `eventSection`→`EventSection`, `eventQuestion`→`EventQuestion`, `EventsPerSessionWithEvent`→`Events/SessionwithEvent` (see the note at the top of this notebook).
2. Drop rows that are blank across every `LS_COLS` field present in the raw data.
3. Drop rows where `Date` is blank.
4. Reformat `Date` to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds).
5. Strip HTML tags/attributes and unescape HTML entities on `EventSection`.
6. Insert a new `Feature` column (always blank — there's no source data for it), positioned between `Operation` and `EventAction` per `LS_COLS`.
7. Reorder/drop columns to match `LS_COLS`.
8. Trim surrounding whitespace, then collapse any internal run of whitespace to a single space, on the `LS_STRING_COLS` fields.
9. Cast `TotalEvents`, `UniqueEvents`, `SessionsWithEvent`, `Events/SessionwithEvent` to integer type.
10. Sort ascending by `Date`, `CompanyCode`, `CompanyName`, `Country`, `HomeCountry`, `HomeCountryCode`.


In [5]:
HTML_TAG_RE = re.compile(r"<[^>]+>")


def strip_html(value: str) -> str:
    text = HTML_TAG_RE.sub("", value)
    text = html.unescape(text)
    return text.replace("\xa0", " ").strip()


def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns=RENAME_MAP)

    # Same reasoning as the other notebooks: judge "completely blank" against the
    # LS_COLS fields present in the raw data, since raw pipeline-metadata columns
    # (DataSource, PipelineRunID, FileName, ...) are dropped later and would otherwise
    # mask genuinely blank rows. Feature isn't inserted yet at this point, so it's
    # naturally excluded from this check.
    present_ls_cols = [c for c in LS_COLS if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    df = df.loc[~is_blank].copy()

    # Only Date being blank drops a row, matching Clean_HubDailyUsers.
    missing_key = df["Date"].str.strip() == ""
    df = df.loc[~missing_key].copy()

    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

    # Strip HTML before the whitespace pass below, so any double spaces left behind by
    # removing tags get collapsed along with genuine source whitespace.
    for col in HTML_COLS:
        df[col] = df[col].apply(strip_html)

    # Feature has no source column in the raw data; the notes ask for it to be inserted
    # as a new column between Operation and EventAction, which LS_COLS already reflects.
    df["Feature"] = ""

    df = df[LS_COLS]

    for col in LS_STRING_COLS:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS:
        df[col] = df[col].astype(int)

    df = df.sort_values(by=SORT_COLS, ascending=True).reset_index(drop=True)

    return df


## Collapse duplicate rows

Duplicates are not allowed in the final cleaned dataset. Rows that share every `GROUP_COLS` value are collapsed into one row, summing `SUM_COLS` (`Users`, `TotalEvents`, `UniqueEvents`, `SessionsWithEvent`, `Events/SessionwithEvent`) as integers.


In [6]:
def collapse_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in SUM_COLS:
        df[col] = df[col].astype(int)
    df = df.groupby(GROUP_COLS, as_index=False)[SUM_COLS].sum()
    return df[LS_COLS]


## Configure the input file

Leave `INPUT_FILE` as `None` to auto-detect the single raw file in this notebook's `input/` folder, or set it to an explicit path to override (equivalent to the script's optional CLI argument).


In [7]:
NOTEBOOK_DIR = Path.cwd()
INPUT_FILE = None  # e.g. "input/HubDailyEventData_2026-08-02.csv"

input_path = Path(INPUT_FILE).resolve() if INPUT_FILE else find_default_input(NOTEBOOK_DIR / "input")
month_tag = month_tag_from_filename(input_path)
input_path, month_tag


(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/HUB/input/HubDailyEventData_2026-08-02.csv'),
 '202607')

## Read the raw CSV

Read everything as strings (`dtype=str`, `keep_default_na=False`) so blank fields and numeric-looking codes pass through unchanged instead of being coerced or turned into `NaN`. `engine="python"` with `escapechar="\\"` (per the notes) matches the other daily notebooks' handling of escaped quotes. `encoding="utf-8"` matches the source file and avoids corrupting accented text.

Before parsing, the raw text is pre-processed to double any backslash that isn't immediately followed by `"`. Without this, `escapechar` strips *every* backslash it precedes, not just ones before a quote — and the raw data contains a real company name, `TBWA\RAAD`, whose backslash isn't a CSV escape artifact at all. The doubling makes `escapechar` only ever consume genuine `\"` sequences, so `TBWA\RAAD` survives intact while the legitimate escaped-quote content (e.g. in `eventLabel`) still parses correctly.


In [8]:
with open(input_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Protect literal backslashes that aren't a genuine CSV \" escape (e.g. the real
# company name "TBWA\RAAD") by doubling them, so escapechar below only ever consumes
# actual \" sequences and every other backslash survives as a literal single backslash.
protected_text = re.sub(r'\\(?!")', r"\\\\", raw_text)

df_raw = pd.read_csv(
    io.StringIO(protected_text), sep=";", engine="python", escapechar="\\",
    dtype=str, keep_default_na=False, encoding="utf-8",
)
df_raw.shape


(73599, 26)

## Apply the cleaning steps

In [9]:
df_cleaned = clean(df_raw)
df_cleaned.head()


,Date,CompanyCode,CompanyName,Country,HomeCountry,HomeCountryCode,Operation,Feature,EventAction,EventCategory,EventLabel,EventSection,EventQuestion,UserType,Users,TotalEvents,UniqueEvents,SessionsWithEvent,Events/SessionwithEvent
0,2026-07-01 00:00:00.000,,,Algeria,,,,,link,,,,,Returning User,1,1,1,1,1
1,2026-07-01 00:00:00.000,,,Australia,,,,,link,,,,,Returning User,1,3,2,1,3
2,2026-07-01 00:00:00.000,,,Austria,,,,,link,,,,,Returning User,1,1,1,1,1
3,2026-07-01 00:00:00.000,,,Bulgaria,,,,,link,,,,,Returning User,1,2,2,1,2
4,2026-07-01 00:00:00.000,,,Canada,,,,,link,,,,,Returning User,1,13,10,3,13


## Collapse duplicate rows before saving

`rows_before_dedup` is kept so the report below can still report "blank/missing-key rows dropped" against the pre-dedup count, separately from rows collapsed for being duplicates.


In [10]:
rows_before_dedup = len(df_cleaned)
df_cleaned = collapse_duplicates(df_cleaned)
duplicates_collapsed = rows_before_dedup - len(df_cleaned)

print(f"Collapsed {duplicates_collapsed} duplicate rows -> {len(df_cleaned)} rows remaining")


Collapsed 1 duplicate rows -> 73596 rows remaining


## Save the cleaned dataset

Written as `;`-delimited UTF-8 with minimal quoting, matching the input file's own semicolon delimiter as required by the notes. Saved to this notebook's `output/` folder.


In [11]:
output_dir = NOTEBOOK_DIR / "output"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"HubDailyEvents_{month_tag}_cleaned.csv"
df_cleaned.to_csv(output_path, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned)} rows -> {output_path}")


Cleaned 73596 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\output\HubDailyEvents_202607_cleaned.csv


## Write summary report

Writes a plain-text report answering: which input file was read, the raw and cleaned row counts, how many duplicate rows exist in each of the raw and cleaned dataframes, the derived `month_tag`, how many rows were dropped for being blank / missing their key fields, and the output CSV's name, followed (after three blank lines) by `df_cleaned.describe()`. Saved to this notebook's `reports/` folder as `HubDailyEvents_<month_tag>_report.txt`.

Duplicate counts use pandas' default `duplicated()` (`keep="first"`), i.e. the number of rows that are repeats of an earlier row — how many rows would go away if the dataframe were deduplicated.


In [12]:
report_lines = [
    f"Input file: {input_path.name}",
    f"Raw row count: {len(df_raw)}",
    f"Raw duplicate rows: {int(df_raw.duplicated().sum())}",
    "=======================================================================",
    f"Month tag: {month_tag}",
    f"Blank/missing-key rows dropped: {len(df_raw) - rows_before_dedup}",
    f"Duplicate rows collapsed: {duplicates_collapsed}",
    "=======================================================================",
    f"Cleaned row count: {len(df_cleaned)}",
    f"Cleaned duplicate rows: {int(df_cleaned.duplicated().sum())}",
    f"Output file: {output_path.name}",
]
report_text = "\n".join(report_lines) + "\n"
report_text += "\n\n\n" + df_cleaned.describe().to_string() + "\n"

reports_dir = NOTEBOOK_DIR / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
report_path = reports_dir / f"HubDailyEvents_{month_tag}_report.txt"
report_path.write_text(report_text, encoding="utf-8")

print(report_text)
print(f"Report written -> {report_path}")


Input file: HubDailyEventData_2026-08-02.csv
Raw row count: 73599
Raw duplicate rows: 0
Month tag: 202607
Blank/missing-key rows dropped: 2
Duplicate rows collapsed: 1
Cleaned row count: 73596
Cleaned duplicate rows: 0
Output file: HubDailyEvents_202607_cleaned.csv



              Users   TotalEvents  UniqueEvents  SessionsWithEvent  Events/SessionwithEvent
count  73596.000000  73596.000000  73596.000000       73596.000000             73596.000000
mean       1.166504      1.905226      1.407169           1.314854                 1.899560
std        1.105789      5.920023      4.051665           2.692513                 5.909761
min        1.000000      1.000000      1.000000           0.000000                 0.000000
25%        1.000000      1.000000      1.000000           1.000000                 1.000000
50%        1.000000      1.000000      1.000000           1.000000                 1.000000
75%        1.000000      2.000000      1.000000           1.000000                 2.00


Report written -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\reports\HubDailyEvents_202607_report.txt
